![image.png](https://i.imgur.com/a3uAqnb.png)

# **🚗 Vehicle Object Detection with YOLOv8**
In this lab, we will:
✅ **Use YOLOv8** for **vehicle detection**  
✅ **Understand the dataset structure**  
✅ **Train a YOLOv8 model**  
✅ **Evaluate the model on the validation set**  
✅ **Run inference on test images**  

---

## **1️⃣ Understanding the Dataset Structure**
The dataset follows the **YOLO format**, which consists of:
📂 **train/** → Training images & labels  
📂 **valid/** → Validation images & labels  
📂 **test/** → Test images (for inference)  
📜 **data.yaml** → Defines dataset paths & class names  

### **🔹 YOLO Dataset Folder Structure**
```
vehicles_dataset/
│── train/
│   │── images/
│   │   ├── pic_031.jpg
│   │   ├── pic_032.jpg
│   │   ├── ...
│   │── labels/
│   │   ├── pic_031.txt
│   │   ├── pic_032.txt
│   │   ├── ...
│
│── valid/
│   │── images/
│   │   ├── pic_035.jpg
│   │   ├── pic_036.jpg
│   │   ├── ...
│   │── labels/
│   │   ├── pic_035.txt
│   │   ├── pic_036.txt
│   │   ├── ...
│
│── test/
│   │── images/
│   │   ├── pic_040.jpg
│   │   ├── pic_041.jpg
│   │   ├── ...
│   │── labels/
│   │   ├── pic_040.txt
│   │   ├── pic_041.txt
│   │   ├── ...
│
│── data.yaml
```
Each **image** has a **corresponding label** file with the **same name**, but a `.txt` extension.

---

## **2️⃣ Loading & Preparing the Dataset**

Now that we understand the layout, let's get the data onto the machine. In this section we will:

- ⬇️ **Download** the vehicle dataset from Google Drive and unzip it
- 📜 **Inspect** `data.yaml` to confirm the class names and split paths
- ✂️ **Re-split** the images into clean train / validation / test sets

In [ ]:
!gdown '1EkA7wJzGnoC0keWA7qUTxe0KGXVaiXtf'

!unzip -q vehicle-detection.v3i.yolov8.zip -d /content/yolo_dataset

!ls /content/yolo_dataset

In [ ]:
# Load dataset configuration
path = "/content/yolo_dataset"
dataset_path = path + "/data.yaml"

# Check dataset information
print(open(dataset_path).read())

### Re-splitting the data

The next cell carves a fresh **validation** (20%) and **test** (10%) set out of the training images, moving each image together with its matching `.txt` label so the splits stay consistent. Run it once.

In [ ]:
# Run this as is (This is to sort out the dataset into train, val, and test segments)

import os
import random
import shutil

# 1. Define all paths
dataset_dir = '/content/yolo_dataset'
train_img_dir = os.path.join(dataset_dir, 'train/images')
train_lbl_dir = os.path.join(dataset_dir, 'train/labels')

val_img_dir = os.path.join(dataset_dir, 'valid/images')
val_lbl_dir = os.path.join(dataset_dir, 'valid/labels')

test_img_dir = os.path.join(dataset_dir, 'test/images')
test_lbl_dir = os.path.join(dataset_dir, 'test/labels')

# 2. Create the new directories
for directory in [val_img_dir, val_lbl_dir, test_img_dir, test_lbl_dir]:
    os.makedirs(directory, exist_ok=True)

# 3. Get all images currently in the train folder
images = os.listdir(train_img_dir)
total_images = len(images)

# 4. Calculate sizes (10% Valid, 10% Test)
val_size = int(total_images * 0.2)
test_size = int(total_images * 0.1)

# 5. Randomly select the images
val_images = random.sample(images, val_size)
# Remove the validation images from the pool so they aren't picked for the test set
remaining_images = [img for img in images if img not in val_images]
test_images = random.sample(remaining_images, test_size)

# 6. Helper function to move the images and their text files
def move_files(file_list, dest_img_dir, dest_lbl_dir):
    for img in file_list:
        # Move image
        shutil.move(os.path.join(train_img_dir, img), os.path.join(dest_img_dir, img))

        # Move corresponding label (swap image extension with .txt)
        lbl = os.path.splitext(img)[0] + '.txt'
        lbl_path = os.path.join(train_lbl_dir, lbl)
        if os.path.exists(lbl_path):
            shutil.move(lbl_path, os.path.join(dest_lbl_dir, lbl))

# 7. Execute the moves
move_files(val_images, val_img_dir, val_lbl_dir)
move_files(test_images, test_img_dir, test_lbl_dir)

# 8. Print the results
print(f"Original total: {total_images} images")
print(f"Moved {len(val_images)} to Validation.")
print(f"Moved {len(test_images)} to Test.")
print(f"Remaining in Train: {len(os.listdir(train_img_dir))}")

## **3️⃣ Setting Up YOLOv8**

Before training, we install the **Ultralytics** library (which provides YOLOv8) and load a **pretrained** `yolov8n.pt` model. Starting from COCO-pretrained weights lets us *fine-tune* rather than train from scratch, which is faster and needs far less data.

In [ ]:
# Install Ultralytics library which has Yolo
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
# Load YOLOv8 model (small version)
model = YOLO("yolov8n.pt")

## **4️⃣ Training a YOLOv8 Model**

We will fine-tune the pretrained model on our six vehicle classes. But first, it helps to know our **starting point** — how does the off-the-shelf model do *before* it has seen a single vehicle image from our dataset? This gives us a baseline to measure improvement against.

### Sanity check: predict on a test image *before* training

The model was pretrained on COCO (which includes some vehicle-like classes), so it may detect a few objects — but don't expect it to recognise our specific categories well yet.

In [ ]:
# Load an image and run inference
from pathlib import Path
test_dir = Path(path + "/test/images")
first_file_path = next((x for x in test_dir.iterdir() if x.is_file()), None)

results = model(first_file_path, save=True)

# Convert result to a NumPy array and display
predicted_image = results[0].plot()  # Convert prediction to an image

plt.figure(figsize=(16, 16))
plt.imshow(predicted_image)
plt.axis("off")
plt.title("Predicted Image")
plt.show()


### Baseline metrics *before* training

A single image is qualitative. To get a number we can compare against later, we run validation on the **pretrained** model. Because `yolov8n.pt` was trained on COCO — not our six vehicle classes — expect low scores here. This is our **"before" reference point**.

In [ ]:
# === Baseline metrics BEFORE training ===
# yolov8n.pt is pretrained on COCO, not our 6 vehicle classes, so expect low
# scores. This is the "before" reference point.
baseline_metrics = model.val(data=dataset_path, verbose=False)

baseline = {
    "precision": baseline_metrics.box.mp,
    "recall":    baseline_metrics.box.mr,
    "mAP50":     baseline_metrics.box.map50,
    "mAP50-95":  baseline_metrics.box.map,
}
print("Baseline (pretrained, before fine-tuning):")
for k, v in baseline.items():
    print(f"  {k:10s}: {v:.4f}")

### Fine-tuning on our vehicle classes

With a baseline recorded, let's train. We first confirm a GPU is available (training on CPU would be very slow), then fine-tune for a few epochs using the Adam optimizer.

In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())

In [ ]:
# Train on the vehicle detection dataset
model.train(data=dataset_path, epochs=20, imgsz=640, batch=32
            , optimizer='Adam', lr0=0.005)

## **5️⃣ Evaluating the Model**
We use **mAP@0.5:0.95** to assess performance.

In [ ]:
# Run validation
metrics = model.val(data=dataset_path)

## **6️⃣ Running Inference on Test Images**


In [ ]:
import glob, os

# Find the most recently written best.pt under runs/detect
best = max(glob.glob("/content/runs/detect/*/weights/best.pt"), key=os.path.getmtime)
print("Using weights:", best)

model = YOLO(best)
results = model("/content/yolo_dataset/test/images/DSC_0015_JPG.rf.2eea7464c126b796b62bc41d00e32a63.jpg", save=True)

predicted_image = results[0].plot()
plt.figure(figsize=(8, 8))
plt.imshow(predicted_image)
plt.axis("off")
plt.title("Predicted Image")
plt.show()

## **7️⃣ Before vs After: Did Fine-Tuning Help?**

Finally, we put the **baseline** (pretrained) and **fine-tuned** metrics side by side — as a table and a grouped bar chart — to see exactly how much training on our vehicle dataset improved precision, recall, and mAP.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

trained = {
    "precision": metrics.box.mp,
    "recall":    metrics.box.mr,
    "mAP50":     metrics.box.map50,
    "mAP50-95":  metrics.box.map,
}

labels = ["precision", "recall", "mAP50", "mAP50-95"]
before = [baseline[k] for k in labels]
after  = [trained[k]  for k in labels]

# Text table
print(f"{'Metric':<12}{'Before':>10}{'After':>10}{'Delta':>10}")
print("-" * 42)
for k, b, a in zip(labels, before, after):
    print(f"{k:<12}{b:>10.4f}{a:>10.4f}{a-b:>+10.4f}")

# Grouped bar chart
x, w = np.arange(len(labels)), 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, before, w, label="Before (pretrained)")
ax.bar(x + w/2, after,  w, label="After (fine-tuned)")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Score"); ax.set_ylim(0, 1)
ax.set_title("YOLOv8n vehicle detection: before vs after fine-tuning")
ax.legend()
for i, (b, a) in enumerate(zip(before, after)):
    ax.text(i - w/2, b + 0.01, f"{b:.2f}", ha="center", va="bottom", fontsize=8)
    ax.text(i + w/2, a + 0.01, f"{a:.2f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.show()

### 🚀 **You now have a working YOLOv8 object detection pipeline for vehicle detection!**

To recap, we:

1. Explored the YOLO-format dataset and its six vehicle classes
2. Downloaded the data and split it into train / validation / test sets
3. Loaded a COCO-pretrained YOLOv8n model and recorded a **baseline**
4. Fine-tuned the model on our vehicle classes
5. Evaluated it with **mAP@0.5:0.95**
6. Ran inference on unseen test images
7. Compared **before vs after** to confirm fine-tuning paid off

**Where to go next:** try more epochs, a larger backbone (`yolov8s`/`yolov8m`), image augmentation, or tuning the learning rate to push the metrics higher.

### Contributed by: Abdulaziz Alomair